# Pydantic and Databases

- Why data validation matters
- Create Pydantic models with `BaseModel`
- Add rules with `Field`
- Handle optional and restricted values
- Validate dictionaries and JSON
- Convert models back to dictionaries
- Understand how Python reads and writes database records
- Connect Pydantic validation to a PostgreSQL workflow

We'll use familiar **Customer, Product, Order, and Support Ticket** examples. Start at the top and run cells in order.

> **Setup:** Install Pydantic for the first sections with `pip install pydantic`. PostgreSQL sections are optional and require a running PostgreSQL database.

## 1. Why do we need validation?

Imagine an application receives a customer record from an API or an AI model. A plain dictionary can contain missing fields or incorrect values.

In [3]:
customer_data = {
    "customer_id": "1001",
    "name": "Bhuvan Tej",
    "email": "tej@example.com"
}

print(customer_data["name"])
print("ID type:", type(customer_data["customer_id"]).__name__)

Bhuvan Tej
ID type: str


The ID is a string, even though our application may expect an integer. We also have no built-in rule that checks whether required fields exist.

**Pydantic** lets us describe the shape of the data and validate incoming values against it.

| | Plain dict + manual checks | Pydantic `BaseModel` |
|-|---------------------------|---------------------|
| Type conversion | Manual | Automatic (`'4'` → `4`) |
| Missing fields | Manual check | Raises `ValidationError` |
| Invalid values | Manual check | Raises `ValidationError` |
| IDE autocomplete | None | Full autocomplete on fields |
| Error messages | You write them | Pydantic generates field-level messages |

## 2. Create your first model

A Pydantic model is a class that inherits from `BaseModel`. The annotations describe the expected fields and types.

In [11]:
from pydantic import BaseModel, ValidationError

# The simplest Pydantic model — just field name and type
class Customer(BaseModel):
    customer_id: int
    name: str
    email: str

customer = Customer(customer_id=1001, name="Bhuvan Tej", email="tej@xyz.com")
print(customer.name)
print(customer.customer_id)
print("ID type:", type(customer.customer_id).__name__)

# Works: Pydantic converts "1001" (str) to 1001 (int)
customer = Customer(customer_id="1001", name='Bhuvan Tej', email='tej@xyz.com')

# Raises ValidationError: "1001a" cannot be converted to an integer
# customer = Customer(customer_id="1001a", name='Bhuvan Tej', email='tej@xyz.com')

print(repr(customer))
print(str(customer))

Bhuvan Tej
1001
ID type: int
Customer(customer_id=1001, name='Bhuvan Tej', email='tej@xyz.com')
customer_id=1001 name='Bhuvan Tej' email='tej@xyz.com'


Pydantic converts `"1001"` to the integer `1001` because the conversion is valid. It raises `ValidationError` when it cannot validate the input.

In [12]:
try:
    invalid_customer = Customer(
        customer_id="not-a-number",
        name="Bhuvan Tej",
        email="tej@xyz.com"
    )
except ValidationError as error:
    print(error)

1 validation error for Customer
customer_id
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/int_parsing


## 3. Add rules with `Field`

`Field()` lets you add rules beyond just the type — min/max values, string lengths, descriptions.

| Constraint | Meaning | Example |
|-----------|---------|--------|
| `gt=0` | greater than 0 | `customer_id: int = Field(gt=0)` |
| `ge=0` | greater than or equal to 0 | `stock: int = Field(ge=0)` |
| `lt=n` | less than n | `rating: int = Field(lt=6)` |
| `le=5` | less than or equal to 5 | `rating: int = Field(le=5)` |
| `min_length=1` | string must have at least 1 char | `name: str = Field(min_length=1)` |
| `max_length=50` | string must have at most 50 chars | `name: str = Field(max_length=50)` |
| `default=value` | default if not provided | `country: str = Field(default='India')` |

Types alone may not be enough. For example, a customer ID should be positive and a name should not be empty.

For production email validation, use Pydantic's email validation support (`EmailStr`) and install its email extra. The simple `email` field above focuses on the core model concepts.

In [1]:
from pydantic import BaseModel, Field

class CustomerRecord(BaseModel):
    customer_id: int = Field(gt=0)
    name: str = Field(min_length=1, max_length=60)
    email: str = Field(min_length=5)
    city: str = "Bengaluru"

good_customer = CustomerRecord(
    customer_id=1001,
    name="Bhuvan Tej",
    email="tej@xyz.com"
)

print(good_customer)
print("City default:", good_customer.city)

customer_id=1001 name='Bhuvan Tej' email='tej@xyz.com' city='Bengaluru'
City default: Bengaluru


In [18]:
try:
    customer = CustomerRecord(
        customer_id=-1,
        name="",
        email="bad"
    )
except ValidationError as e:
    print('ValidationError — field-level details:')
    for err in e.errors():
        print(f"  field: {err['loc']}  →  {err['msg']}")

ValidationError — field-level details:
  field: ('customer_id',)  →  Input should be greater than 0
  field: ('name',)  →  String should have at least 1 character
  field: ('email',)  →  String should have at least 5 characters


## 4. Optional fields and allowed values

`Optional` — field that can be `None`
- `Optional[str]` means the field can be a `str` OR `None`. Use it for fields that might not always exist.

`Literal` — restrict to specific allowed values
- `Literal['a', 'b', 'c']` means the value must be exactly one of those strings. If returns anything else → `ValidationError`.

Some fields may be absent. Other fields should accept only specific values. For example, an order status might be `Pending`, `Shipped`, or `Delivered`.

In [23]:
from typing import Literal

class Order(BaseModel):
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    total_amount: float = Field(ge=0)
    status: Literal["Pending", "Shipped", "Delivered", "Cancelled"]
    notes: str | None = None

order = Order(
    order_id=5001,
    customer_id=1001,
    total_amount=2499.50,
    status="Shipped"
)

print(order)
print("Notes:", order.notes)

try:
    Order(
        order_id=5002,
        customer_id=1001,
        total_amount=2499.50,
        status="On the way"  # not one of the allowed values
    )
except ValidationError as e:
    print()
    print('Invalid status caught:')
    for err in e.errors():
        print(f"  {err['loc']}  →  {err['msg']}")

order_id=5001 customer_id=1001 total_amount=2499.5 status='Shipped' notes=None
Notes: None

Invalid status caught:
  ('status',)  →  Input should be 'Pending', 'Shipped', 'Delivered' or 'Cancelled'


`str | None = None` means the notes field may contain a string or `None`, and it is optional because it has a default.

In [2]:
from typing import Optional, Literal

class ProductRecord(BaseModel):
    product_id    : int   = Field(gt=0)
    product_name  : str   = Field(min_length=1)
    category      : str
    price         : float = Field(gt=0)
    stock_quantity: int   = Field(ge=0)           # 0 is valid (out of stock)
    description   : Optional[str] = None          # may or may not exist

    @property
    def is_in_stock(self):
        return self.stock_quantity > 0

    @property
    def price_str(self):
        return f'${self.price:.2f}'

    def __repr__(self):
        return f'ProductRecord(id={self.product_id}, name={self.product_name!r}, price={self.price_str})'


# description is None — that is fine
p1 = ProductRecord(product_id=101, product_name='Monitor',
                   category='Electronics', price=205.21, stock_quantity=238)
print(p1)
print('in stock:', p1.is_in_stock)
print('description:', p1.description)   # None

product_id=101 product_name='Monitor' category='Electronics' price=205.21 stock_quantity=238 description=None
in stock: True
description: None


## 5. Validate dictionaries and JSON

Incoming data often arrives as a Python dictionary or JSON text.

In [3]:
from pydantic import Field

# Validate a Python dictionary, such as an API or database record.
class Order(BaseModel):
    order_id: int = Field(gt=0)
    customer_id: int = Field(gt=0)
    total_amount: float = Field(ge=0)
    status: Literal["Pending", "Shipped", "Delivered", "Cancelled"]
    notes: str | None = None

raw_order = {
    "order_id": "5003",
    "customer_id": "1001",
    "total_amount": "799.99",
    "status": "Pending",
    "notes": "Leave at reception"
}

order_from_dict = Order.model_validate(raw_order)
print(order_from_dict)
print(type(order_from_dict.total_amount).__name__)

order_id=5003 customer_id=1001 total_amount=799.99 status='Pending' notes='Leave at reception'
float


In [4]:
# Validate JSON text in one step.
order_json = '''
{
  "order_id": 5004,
  "customer_id": 1002,
  "total_amount": 1299.0,
  "status": "Delivered",
  "notes": "Leave at reception"
}
'''

order_from_json = Order.model_validate_json(order_json)
print(order_from_json)

order_id=5004 customer_id=1002 total_amount=1299.0 status='Delivered' notes='Leave at reception'


## 6. Convert a model back to data

`model_dump()` converts a Pydantic object back into a plain Python dictionary. Use `model_dump_json()` when you need JSON text.

In [5]:
order_dict = order_from_json.model_dump()
order_json_output = order_from_json.model_dump_json()

print(type(order_dict).__name__, order_dict)
print(type(order_json_output).__name__, order_json_output)

dict {'order_id': 5004, 'customer_id': 1002, 'total_amount': 1299.0, 'status': 'Delivered', 'notes': 'Leave at reception'}
str {"order_id":5004,"customer_id":1002,"total_amount":1299.0,"status":"Delivered","notes":"Leave at reception"}


## 7. A practical AI Engineering example: validate an AI classification

An AI model might classify a customer support message. Do not blindly trust its output: validate the category and required fields before using them.

In [6]:
class SupportTicketResult(BaseModel):
    category: Literal["TRACK_ORDER", "CANCEL_ORDER", "REFUND", "GENERAL"]
    confidence: Literal["low", "medium", "high"]
    reason: str = Field(min_length=5)

ai_json = '''
{
  "category": "TRACK_ORDER",
  "confidence": "high",
  "reason": "The customer asks where their order is."
}
'''

ticket_result = SupportTicketResult.model_validate_json(ai_json)
print("Category:", ticket_result.category)
print("Confidence:", ticket_result.confidence)
print("Reason:", ticket_result.reason)

Category: TRACK_ORDER
Confidence: high
Reason: The customer asks where their order is.


Try changing `category` to `"OTHER"` or removing `reason`. Run the cell again to see validation catch the problem.

**Important:** validation checks structure and rules; it does not prove that the AI's answer is factually correct.

## 8. Nested models: an employee with an address

Real records can contain nested objects and lists. Pydantic can validate these structures too.

In [19]:
class Address(BaseModel):
    city: str
    state: str

class Employee(BaseModel):
    employee_id: int = Field(gt=0)
    name: str
    address: Address
    skills: list[str]

employee_data = {
    "employee_id": 101,
    "name": "Ananya",
    "address": {"city": "Hyderabad", "state": "TS"},
    "skills": ["Python", "SQL", "AWS"]
}

employee = Employee.model_validate(employee_data)
print(employee.name)
print(employee.address.city)
print(employee.skills)

Ananya
Hyderabad
['Python', 'SQL', 'AWS']


In [20]:
class Address(BaseModel):
    city: str
    state: str

class Employee(BaseModel):
    employee_id:int = Field(gt=0)
    name:str = Field(min_length=1, max_length=10)
    address: Address

employee_json = {
    "employee_id": 101,
    "name": "Ananya",
    "address": {"city": "Hyderabad", "state": "TS"}
}

Employee.model_validate(employee_json)

Employee(employee_id=101, name='Ananya', address=Address(city='Hyderabad', state='TS'))

## 9. Database basics: where Pydantic fits

A database stores records. Python sends SQL commands through a database driver.

- **Connection:** opens a session with the database.
- **Cursor:** executes SQL and reads results.
- **INSERT:** adds a row.
- **SELECT:** reads rows.
- **Commit:** saves changes made by a transaction.
- **Parameterized query:** passes values separately from SQL, helping prevent SQL injection.

The flow we want is:

`Database row → Python dictionary → Pydantic validation → application logic`

### Optional PostgreSQL setup

This section requires a PostgreSQL server and these packages:

```bash
pip install "psycopg[binary]" python-dotenv
```

Create a `.env` file (keep it private and never commit it to Git):

```text
DB_HOST=localhost
DB_PORT=5432
DB_NAME=shopsmart
DB_USER=postgres
DB_PASSWORD=your_password
```

The next cells do **not** connect automatically. Run them only after setting up your own database and credentials.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [3]:
# Import os to access environment variables.
# Environment variables allow us to store database credentials
# outside the Python source code.
import os

# Import psycopg, the PostgreSQL database adapter for Python (version 3).
import psycopg


# Function to establish a connection to the PostgreSQL database.
def get_connection():

    # Define the mandatory database configuration settings.
    # These values must be available as environment variables.
    required = ["DB_NAME", "DB_USER", "DB_PASSWORD"]

    # Check whether any required environment variables are missing.
    # os.getenv() returns None if the environment variable doesn't exist
    # or an empty string if it has no value.
    missing = [key for key in required if not os.getenv(key)]

    # If any mandatory settings are missing, raise an error.
    # This prevents attempting a database connection with incomplete
    # configuration.
    if missing:
        raise ValueError(
            f"Missing database settings: {', '.join(missing)}"
        )

    # Establish and return a connection to PostgreSQL.
    return psycopg.connect(

        # Database server hostname.
        # If DB_HOST is not configured, use localhost.
        host=os.getenv("DB_HOST", "localhost"),

        # PostgreSQL server port.
        # If DB_PORT is not configured, use the default port 5432.
        # Convert the port from a string to an integer.
        port=int(os.getenv("DB_PORT", "5432")),

        # Name of the database to connect to.
        dbname=os.getenv("DB_NAME"),

        # Database username.
        user=os.getenv("DB_USER"),

        # Database password.
        password=os.getenv("DB_PASSWORD"),
    )

# Print a message confirming that the connection helper is defined.
# Note: This does not establish an actual database connection.
print("Connection helper defined. Connect only when PostgreSQL is configured.")

Connection helper defined. Connect only when PostgreSQL is configured.


In [4]:
# STEP 1: Establish a connection to the PostgreSQL database.
# get_connection() returns a connection object.
# This object represents an active connection between Python and PostgreSQL.
conn = get_connection()


# STEP 2: Create a cursor from the connection object.
# A cursor is used to execute SQL queries and interact with the database.
cur = conn.cursor()

# STEP 3: Execute the SQL query using the cursor.
# CREATE TABLE IF NOT EXISTS creates the customers table
# only if it doesn't already exist.
cur.execute("""
    CREATE TABLE IF NOT EXISTS customers (
        customer_id INTEGER PRIMARY KEY,
        name VARCHAR(60) NOT NULL,
        email VARCHAR(120) NOT NULL,
        city VARCHAR(60) NOT NULL
    )
""")


# STEP 4: Commit the transaction.
# commit() permanently saves the changes made during this transaction.
# Without committing, the changes may not be saved.
conn.commit()

# STEP 5: Close the database connection.
# Closing the connection releases database resources.
conn.close()

<psycopg.Cursor [COMMAND_OK] [INTRANS] (host=localhost database=postgres) at 0x112b84300>

In [6]:
# Function to create a customers table in the PostgreSQL database.
def create_customers_table():

    # Establish a connection to the database.
    # The connection is automatically closed when the with block exits.
    with get_connection() as conn:

        # Create a cursor to execute SQL statements.
        with conn.cursor() as cur:

            # Execute the CREATE TABLE SQL statement.
            cur.execute("""
                CREATE TABLE IF NOT EXISTS customers (

                    -- INTEGER stores whole numbers.
                    -- PRIMARY KEY uniquely identifies each customer.
                    -- Duplicate customer_id values are not allowed.
                    customer_id INTEGER PRIMARY KEY,

                    -- VARCHAR(60) stores text up to 60 characters.
                    -- NOT NULL means this column cannot contain NULL values.
                    name VARCHAR(60) NOT NULL,

                    -- Stores the customer's email address.
                    -- Maximum allowed length is 120 characters.
                    email VARCHAR(120) NOT NULL,

                    -- Stores the customer's city.
                    -- Maximum allowed length is 60 characters.
                    city VARCHAR(60) NOT NULL
                )
            """)

        # When the connection context exits successfully,
        # the transaction is automatically committed.
        # If an exception occurs, the transaction is rolled back.

    # Print a confirmation message after table creation.
    print("Customers table is ready.")

# Call the function to create the customers table.
# If the table already exists, PostgreSQL leaves it unchanged.
create_customers_table()

Customers table is ready.


In [8]:
# Function to insert a new customer record into the customers table.
def insert_customer(customer_id, name, email, city):

    # Establish a connection to the database.
    # The connection is automatically closed when the with block exits.
    with get_connection() as conn:

        # Create a cursor to execute SQL statements.
        with conn.cursor() as cur:

            # Execute the INSERT SQL statement.
            #
            # %s represents placeholders for the values we want to insert.
            # We pass the actual values separately as a tuple.
            #
            # This is called a parameterized query.
            # It helps prevent SQL injection and avoids manually
            # constructing SQL strings using f-strings.
            cur.execute(
                """
                INSERT INTO customers (customer_id, name, email, city)
                VALUES (%s, %s, %s, %s)

                -- If a customer with the same customer_id already exists,
                -- do nothing instead of raising a duplicate key error.
                ON CONFLICT (customer_id) DO NOTHING
                """,

                # Actual values supplied to the SQL placeholders.
                (customer_id, name, email, city)
            )

        # When the connection context exits successfully,
        # the transaction is automatically committed.
        # If an exception occurs, the transaction is rolled back.


    # Print a confirmation message after the database operation.
    print("Insert checked for customer:", customer_id)


# Call the function to insert a new customer.
# This inserts customer 1002 if that customer_id does not already exist.
insert_customer(
    1002,
    "Prudhvi Akella",
    "prudhvi@example.com",
    "Hyderabad"
)

Insert checked for customer: 1002


In [9]:
# Function to fetch a single customer using their customer_id.
def get_customer(customer_id):

    # Establish a connection to the database.
    # The connection is automatically closed when the with block exits.
    with get_connection() as conn:

        # Create a cursor to execute SQL queries.
        with conn.cursor() as cur:

            # Execute the SQL query to fetch a specific customer.
            # %s acts as a placeholder for the customer_id value.
            # (customer_id,) passes the value as a tuple.
            #
            # Using parameterized queries helps prevent SQL injection.
            cur.execute(
                "SELECT customer_id, name, email, city FROM customers WHERE customer_id = %s",
                (customer_id,)
            )

            # fetchone() retrieves only the first matching record.
            # By default, the returned record is a tuple.
            # If no matching record exists, it returns None.
            row = cur.fetchone()

    # Check whether a customer was found.
    # If no matching record exists, return None.
    if row is None:
        return None

    # Convert the returned tuple into a dictionary.
    # Each tuple position corresponds to a selected database column.
    return {
        "customer_id": row[0],
        "name": row[1],
        "email": row[2],
        "city": row[3]
    }


# Call the function to retrieve the customer with ID 1001.
row = get_customer(1001)

# Print the retrieved customer dictionary.
print(row)


# Optional: Validate the retrieved dictionary using Pydantic.
# First, check whether a customer was found.
# if row is not None:

#     # model_validate() validates the dictionary against CustomerRecord.
#     # It checks the expected fields and data types and creates
#     # a Pydantic model object.
#     validated_customer = CustomerRecord.model_validate(row)

#     # Print the validated Pydantic object.
#     print(validated_customer)

{'customer_id': 1001, 'name': 'Ravi Kumar', 'email': 'ravi@example.com', 'city': 'Hyderabad'}


In [13]:
def get_customers():

    # Establish a connection to the database.
    # The connection is automatically closed when the with block exits.
    with get_connection() as conn:

        # Create a cursor to execute SQL queries and retrieve results.
        with conn.cursor() as cur:

            # Execute the SQL query to fetch customer details.
            cur.execute(
                "SELECT customer_id, name, email, city FROM customers",
            )

            # fetchall() retrieves all matching records from the database.
            # Each record is returned as a tuple by default.
            # list() converts the result into a Python list.
            rows = list(cur.fetchall())

            # Convert each tuple into a dictionary.
            # row[0] -> customer_id
            # row[1] -> name
            # row[2] -> email
            # row[3] -> city
            #
            # This makes the data easier to access using column names
            # instead of remembering column positions.
            return [
                {
                    "customer_id": row[0],
                    "name": row[1],
                    "email": row[2],
                    "city": row[3]
                }
                for row in rows
            ]

# Call the function to retrieve customer records from the database.
rows = get_customers()
# Print the list of customer dictionaries.
print(rows)

[{'customer_id': 1001, 'name': 'Ravi Kumar', 'email': 'ravi@example.com', 'city': 'Hyderabad'}, {'customer_id': 1002, 'name': 'Prudhvi Akella', 'email': 'prudhvi@example.com', 'city': 'Hyderabad'}]


In [17]:
# Import BaseModel from Pydantic.
# BaseModel allows us to define a data structure with type validation.
from pydantic import BaseModel

# Import dict_row from psycopg.
# dict_row makes the database cursor return each row as a dictionary
# instead of a tuple.
from psycopg.rows import dict_row


def get_customers():
    # Establish a database connection.
    # The connection is automatically closed when we exit the 'with' block.
    with get_connection() as conn:

        # Create a cursor that returns database records as dictionaries.
        with conn.cursor(row_factory=dict_row) as cur:

            # Execute the SQL query to fetch customer details.
            cur.execute(
                "SELECT customer_id, name, email, city FROM customers"
            )

            # Fetch all records and return them as a list of dictionaries.
            return cur.fetchall()


# Call the function to retrieve customer records from the database.
rows = get_customers()


# Define a Customer model using Pydantic.
# This model specifies the expected structure and data types
# for every customer record.
class Customer(BaseModel):
    customer_id: int
    name: str
    email: str
    city: str


# Convert each dictionary into a Customer Pydantic object.
# model_validate() validates the incoming data against the model.
# If the data types are compatible, Pydantic creates a Customer object.
# Otherwise, it raises a validation error.
customer_objects = [
    Customer.model_validate(row) for row in rows
]

# Print the list of validated Customer objects.
print(customer_objects)

[Customer(customer_id=1001, name='Ravi Kumar', email='ravi@example.com', city='Hyderabad'), Customer(customer_id=1002, name='Prudhvi Akella', email='prudhvi@example.com', city='Hyderabad')]


## 10. Practice tasks

Try these before looking at the summary.

1. Create a `Product` model with `product_id`, `name`, `price`, and `stock`.
2. Require `price` to be greater than zero and `stock` to be zero or greater.
3. Add a `status` field to an `Order` that allows only `Pending`, `Shipped`, or `Delivered`.
4. Validate an order from a dictionary where numeric values arrive as strings.
5. Create a `SupportTicketResult` with an invalid category and inspect the error.
6. Create an `Employee` model with a nested `Address` and a list of skills.
7. Explain why SQL values should be passed as parameters rather than inserted into a query using an f-string.

## Day 08 recap

| Concept | Remember |
|---|---|
| `BaseModel` | Defines a validated data structure |
| Type annotations | Describe expected field types |
| `Field(...)` | Adds constraints and defaults |
| `Literal[...]` | Restricts values to an allowed set |
| `ValidationError` | Explains why input failed validation |
| `model_validate(dict)` | Validates a Python dictionary |
| `model_validate_json(text)` | Parses and validates JSON |
| `model_dump()` | Converts a model to a dictionary |
| SQL parameters | Keep data values separate from SQL text |
| Database + Pydantic | Validate records before application logic |

**Next:** Day 09 — HTTP requests and APIs.